In [1]:
import xarray as xr
import os   
import parflow as pf
import plotly.express as px
from parflow.tools.hydrology import calculate_overland_flow_grid, calculate_subsurface_storage, calculate_water_table_depth
import numpy as np
import shutil
import json
import plotly.io as pio
import matplotlib.pyplot as mpl
import math

In [2]:
root_dir = "/glade/derecho/scratch/bwest/drought-ensemble"
domain = "wolf"
ensemble_name = "ensemble_1"
ensemble_member = "8_year_drought"
files = json.load(open(f"{root_dir}/domains/{domain}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"))
drought = xr.open_mfdataset(files, concat_dim="time", combine="nested")
#reindex the time dimension to follow the index
drought["time"] = drought["time"] + (xr.ufuncs.floor(drought["time"] / 8760.) * 8760.) 
drought.info()

/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'cfradial1' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'furuno' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gamic' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/bwest/conda-envs/droughts/share/proj failed
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 

xarray.Dataset {
dimensions:
	time = 140160 ;
	z = 10 ;
	y = 41 ;
	x = 78 ;

variables:
	float64 pressure(time, z, y, x) ;
	float64 saturation(time, z, y, x) ;
	float64 evaptrans(time, z, y, x) ;
	float64 overland_bc_flux(time, y, x) ;
	float64 mask(time, z, y, x) ;
	float64 mannings(time, y, x) ;
	float64 porosity(time, z, y, x) ;
	float64 specific_storage(time, z, y, x) ;
	float64 DZ_Multiplier(time, z, y, x) ;
	float64 slopex(time, y, x) ;
	float64 slopey(time, y, x) ;
	float64 perm_x(time, z, y, x) ;
	float64 perm_y(time, z, y, x) ;
	float64 perm_z(time, z, y, x) ;
	float64 overland_flow(time, y, x) ;
	float64 subsurface_storage(time, z, y, x) ;
	float64 time(time) ;

// global attributes:
}

In [3]:
drought.time

<xarray.DataArray 'time' (time: 140160)> Size: 1MB
array([1.000e+00, 2.000e+00, 3.000e+00, ..., 8.758e+03, 8.759e+03, 1.752e+04])
Coordinates:
  * time     (time) float64 1MB 1.0 2.0 3.0 ... 8.758e+03 8.759e+03 1.752e+04

In [4]:
root_dir = "/glade/derecho/scratch/bwest/drought-ensemble"
domain = "wolf"
ensemble_name = "ensemble_1"
ensemble_member = "baseline"
files = json.load(open(f"{root_dir}/domains/{domain}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"))
baseline = xr.open_mfdataset(files, concat_dim="time", combine="nested")
baseline.info()

/glade/derecho/scratch/bwest/tmp/ipykernel_60076/2191563226.py:6: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  baseline = xr.open_mfdataset(files, concat_dim="time", combine="nested")


xarray.Dataset {
dimensions:
	time = 96360 ;
	z = 10 ;
	y = 41 ;
	x = 78 ;

variables:
	float64 pressure(time, z, y, x) ;
	float64 saturation(time, z, y, x) ;
	float64 evaptrans(time, z, y, x) ;
	float64 overland_bc_flux(time, y, x) ;
	float64 mask(time, z, y, x) ;
	float64 mannings(time, y, x) ;
	float64 porosity(time, z, y, x) ;
	float64 specific_storage(time, z, y, x) ;
	float64 DZ_Multiplier(time, z, y, x) ;
	float64 slopex(time, y, x) ;
	float64 slopey(time, y, x) ;
	float64 perm_x(time, z, y, x) ;
	float64 perm_y(time, z, y, x) ;
	float64 perm_z(time, z, y, x) ;
	float64 overland_flow(time, y, x) ;
	float64 subsurface_storage(time, z, y, x) ;
	float64 time(time) ;

// global attributes:
}

In [5]:
year_start = 8760*3
year_end = 8760*10
drought_storage = drought["subsurface_storage"].isel(time=slice(year_start, year_end, 24)).sum(dim=["x", "y", "z"])
baseline_storage = baseline["subsurface_storage"].isel(time=slice(year_start, year_end, 24)).sum(dim=["x", "y", "z"])
storage_diff = drought_storage - baseline_storage




In [6]:
# plot a line chart of the storage over time
px.line(storage_diff)


In [7]:
drought_storage.time

<xarray.DataArray 'time' (time: 2555)> Size: 20kB
array([1.000e+00, 2.500e+01, 4.900e+01, ..., 8.689e+03, 8.713e+03, 8.737e+03])
Coordinates:
  * time     (time) float64 20kB 1.0 25.0 49.0 ... 8.689e+03 8.713e+03 8.737e+03

In [8]:
outlet_x = 18
outlet_y = 21

drought_streamflow = drought["overland_flow"].isel(time=slice(year_start, year_end, 24), x=outlet_x, y=outlet_y)
baseline_streamflow = baseline["overland_flow"].isel(time=slice(year_start, year_end, 24), x=outlet_x, y=outlet_y)
streamflow_diff = drought_streamflow - baseline_streamflow
# mean_streamflow_diff =streamflow_diff.rolling(time=24*365, min_periods=1).mean()
# mean_streamflow_diff=streamflow_diff.isel(time=slice(year_start, year_end, 876))

In [9]:
px.line(streamflow_diff)